# MCP: Model Context Protocol

A standard wire format between agents and the tools, resources, and prompts they consume.

| Primitive | What it is | Why  |
|-----------|------------|----------------|
| **Model Context Protocol** | Anthropic's open spec for how LLM hosts talk to tool / data servers | The single contract every agent and every tool can agree on |
| `MCPServer` | A process that exposes tools, resources, and prompts | Where capabilities live; reusable across many clients |
| `MCPClient` | The LLM host that discovers and calls a server's primitives | Where the LLM lives; speaks JSON-RPC to one or more servers |
| **JSON-RPC** | Lightweight request / response envelope used as the transport | Why an MCP tool from team A works inside an agent from team B |
| `@tool` + `bind_tools` | LangChain bridge from MCP descriptors to LLM-callable tools | Lets us reuse the same MCP server inside any LangChain agent |

Every agent today re-invents glue code for every backend it talks to; like its own JSON shape for tools, its own way of listing memory, its own prompt-template handoff. **MCP** is the proposal to standardise that wire format so a tool written once is usable anywhere.

We will not require the official `mcp` package; instead we will build a small **MCP-shaped** Python layer that mirrors the spec so the protocol shape itself.

## Setup

In [5]:
# !pip install -q langchain langchain-google-genai langchain-openai langchain-community sentence_transformers

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.tools import tool
from typing import List, Dict, Any, Optional, Callable
from dataclasses import dataclass, field
from dotenv import load_dotenv
import os, json

load_dotenv()

True

In [6]:
os.environ['GEMINI_API_KEY'] = "" # the variable for API key

# llm   = init_chat_model('gpt-4o', model_provider='openai', temperature=0)
llm   = init_chat_model('gemini-2.5-flash', model_provider='google_genai', temperature=0)

embed = init_embeddings('sentence-transformers/all-MiniLM-L6-v2', provider='huggingface')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## The integration-explosion problem MCP solves

Now let's frame the problem before we build the protocol.

Now, we will
- show what an ad-hoc tool registry looks like when every agent invents its own,
- count the integration cost of N agents talking to M tools,
- name the missing piece that **MCP** fills, and why a single wire format collapses N x M into N + M.

Without a standard, every agent x every tool pair is custom integration code. Agent A wants a calendar tool: write the wrapper. Agent B wants the same calendar tool: write a slightly different wrapper. Team C wants both A's and B's tools plus three of its own: rewrite all five with their own conventions. That is the integration-explosion problem HTTP solved for the web in the 1990s. **MCP solves it for agents.**

In [7]:
# A deliberately ad-hoc tool registry: each tool defines its OWN dict shape.
# Some tools have 'args', others 'inputs'. Some return raw text, others wrap it.
# An agent calling these has to special-case every single one.
AD_HOC_TOOLS = [
    {'name': 'weather',  'args':   {'city': 'str'},                'returns': 'string'},
    {'name': 'fx',       'inputs': ['amount', 'from', 'to'],       'output':  'number'},
    {'name': 'notes',    'parameters': {'kind': 'read', 'path': 'notes.md'}},
]

# Count the glue surface: N agents x M tools x per-tool-shape parsing.
n_agents, m_tools = 4, len(AD_HOC_TOOLS)
print('Ad-hoc tool shapes:', [list(t.keys()) for t in AD_HOC_TOOLS])
print(f'Glue functions needed without a standard: ~{n_agents * m_tools}')
print(f'Glue functions needed WITH MCP          : {n_agents + m_tools}  (each speaks one protocol)')

Ad-hoc tool shapes: [['name', 'args', 'returns'], ['name', 'inputs', 'output'], ['name', 'parameters']]
Glue functions needed without a standard: ~12
Glue functions needed WITH MCP          : 7  (each speaks one protocol)


That collapse from N x M to N + M is the whole pitch. **MCP** publishes one wire format: a JSON-RPC envelope plus three primitives (**tools**, **resources**, **prompts**), that every agent and every tool agrees on. Once we have that, **composability** is free: any MCP-compliant client can use any MCP-compliant server it can reach.

## The MCP architecture: client, server, primitives

Now let's name the moving parts so the rest of the notebook reads cleanly.

Now, we will
- define the three roles in the spec: **client** (LLM host), **server** (capability process), **transport** (JSON-RPC),
- enumerate the three primitives a server can expose: **tools** (callable functions), **resources** (read-only data), **prompts** (templated message lists),
- print a tiny architecture diagram in plain text so the picture stays in the notebook output.


```Text
  +-----------------+        JSON-RPC          +--------------------+
  |    MCPClient    |  <-------------------->  |     MCPServer      |
  | (LLM host: app, |   list_tools / call_tool | tools / resources  |
  |  IDE, agent)    |   list_resources / read  |     / prompts      |
  +-----------------+                          +--------------------+
         ^                                              ^
         | bind_tools, @tool                            | register_tool, register_resource
         |                                              |
      LangChain agent                              In-process Python
```

MCP server primitives
  - tool     : callable function with a JSON Schema for arguments
  - resource : read-only blob addressed by a URI (file://, http://, custom)
  - prompt   : parameterised message-list template the client can render

The shape is intentionally small. A server publishes a catalog; the client introspects it; the LLM picks from the catalog. Because both sides speak **JSON-RPC**, neither has to know the other's language, framework, or hosting model.

## Building an `MCPServer` in pure Python

Let's build a minimal MCP-shaped server. We will not require the official `mcp` package; instead we mirror the spec's method names and return shapes so the protocol itself is the lesson.

Now, we will
- define dataclasses `ToolSpec`, `ResourceSpec`, `PromptSpec` matching the MCP descriptor fields,
- implement an `MCPServer` class with `list_tools`, `call_tool`, `list_resources`, `read_resource`, `list_prompts`,
- have every method return JSON-serialisable dicts, the same shape an HTTP wrapper would put on the wire,
- add a `register_tool` / `register_resource` / `register_prompt` registration API so a server is composable from outside.

In [8]:
@dataclass
class ToolSpec:
    """Mirror of an MCP tool descriptor. The `inputSchema` field is camelCase
    on purpose — that is what the spec uses on the wire, and we want our
    list_tools() output to be drop-in for a real MCP client."""
    name: str
    description: str
    inputSchema: Dict[str, Any]
    handler: Callable[..., Any] = field(repr=False)

@dataclass
class ResourceSpec:
    uri: str
    name: str
    mimeType: str
    reader: Callable[[], str] = field(repr=False)

@dataclass
class PromptSpec:
    name: str
    description: str
    arguments: List[Dict[str, Any]]
    template: str

Note the camelCase `inputSchema` and `mimeType`, those match the on-the-wire field names from the MCP spec. Keeping them faithful means a real MCP client could consume our `list_tools()` output without any translation layer. The class itself is small because the protocol is small.

In [9]:
class MCPServer:
    """Minimal MCP-shaped server. Returns JSON-serialisable dicts that mirror the spec.
    A real server would also speak JSON-RPC over stdio / SSE / HTTP; we keep transport in
    a sibling demo so the SHAPE is isolated from the WIRE."""

    def __init__(self, name: str):
        self.name = name
        self._tools:     Dict[str, ToolSpec]     = {}
        self._resources: Dict[str, ResourceSpec] = {}
        self._prompts:   Dict[str, PromptSpec]   = {}

    def register_tool(self, spec: ToolSpec) -> None:
        self._tools[spec.name] = spec

    def register_resource(self, spec: ResourceSpec) -> None:
        self._resources[spec.uri] = spec

    def register_prompt(self, spec: PromptSpec) -> None:
        self._prompts[spec.name] = spec

    def list_tools(self) -> List[Dict[str, Any]]:
        """Return JSON-serialisable tool descriptors. Mirrors MCP `tools/list` response."""
        return [{'name': t.name, 'description': t.description, 'inputSchema': t.inputSchema}
                for t in self._tools.values()]

    def call_tool(self, name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        """Invoke a registered tool. Mirrors MCP `tools/call`."""
        spec = self._tools[name]
        result = spec.handler(**args)
        return {'tool': name, 'isError': False, 'content': result}

    def list_resources(self) -> List[Dict[str, Any]]:
        """Mirrors MCP `resources/list`. Each entry carries uri / name / mimeType."""
        return [{'uri': r.uri, 'name': r.name, 'mimeType': r.mimeType}
                for r in self._resources.values()]

    def read_resource(self, uri: str) -> Dict[str, Any]:
        """Mirrors MCP `resources/read`. Returns the blob plus its mimeType."""
        spec = self._resources[uri]
        return {'uri': uri, 'mimeType': spec.mimeType, 'text': spec.reader()}

    def list_prompts(self) -> List[Dict[str, Any]]:
        """Mirrors MCP `prompts/list`. Templates surface their parameter list."""
        return [{'name': p.name, 'description': p.description, 'arguments': p.arguments}
                for p in self._prompts.values()]

print('MCPServer class defined.')

MCPServer class defined.


Every method returns plain Python dicts that round-trip through `json.dumps` unchanged. That is the whole point, the same response could be sent over JSON-RPC to a remote process, or returned in-process to a library caller, or cached to disk for replay. The shape is the contract.

## A demo server with two tools, one resource, one prompt

Now let's stand up a `DemoServer` that registers concrete capabilities. This is what an integrator on team A would publish so teams B, C, D can consume it.

Now, we will
- implement two callable tool handlers (`weather_lookup` and `currency_convert`) and register them as `ToolSpec`s with explicit JSON Schemas,
- register a single read-only `ResourceSpec` backed by an in-memory notes blob (mimeType `text/markdown`),
- register a single parameterised `PromptSpec` (`brief_summary`) that any client can render with `{topic}`,
- print the resulting catalog as the server would publish it.

In [10]:
def weather_lookup(city: str) -> str:
    """Return a fake weather string for a city. In production this would call a real API."""
    table = {
        'Mumbai':    'Mumbai: 32C, humid, light rain.',
        'Bengaluru': 'Bengaluru: 24C, partly cloudy.',
        'Delhi':     'Delhi: 38C, clear and dry.',
    }
    return table.get(city, f'{city}: weather unavailable in demo dataset.')

def currency_convert(amount: float, from_ccy: str, to_ccy: str) -> str:
    """Convert `amount` from one currency to another using a tiny static rate table."""
    rates = {('USD', 'INR'): 83.0, ('INR', 'USD'): 1 / 83.0,
             ('USD', 'EUR'): 0.92, ('EUR', 'USD'): 1 / 0.92}
    rate = rates.get((from_ccy, to_ccy))
    if rate is None:
        return f'No rate for {from_ccy}->{to_ccy} in demo table.'
    return f'{amount} {from_ccy} = {amount * rate:.2f} {to_ccy}'

Each handler is a plain Python function — no MCP awareness, no decorators. The server wraps them in a `ToolSpec` and publishes the JSON Schema separately. That separation is what lets the same handler be exposed via MCP today and via a different transport tomorrow without touching the function body.

In [11]:
DemoServer = MCPServer(name='demo-server')

# Tool 1: weather_lookup
DemoServer.register_tool(ToolSpec(
    name='weather_lookup',
    description='Look up the current weather for a city.',
    inputSchema={
        'type': 'object',
        'properties': {'city': {'type': 'string', 'description': 'City name to get weather for, e.g. Mumbai'}},
        'required': ['city'],
    },
    handler=weather_lookup,
))

# Tool 2: currency_convert
DemoServer.register_tool(ToolSpec(
    name='currency_convert',
    description='Convert an amount from one currency to another.',
    inputSchema={
        'type': 'object',
        'properties': {
            'amount':   {'type': 'number'},
            'from_ccy': {'type': 'string', 'description': 'ISO 4217, e.g. USD'},
            'to_ccy':   {'type': 'string', 'description': 'ISO 4217, e.g. INR'},
        },
        'required': ['amount', 'from_ccy', 'to_ccy'],
    },
    handler=currency_convert,
))

# Resource: a small notes blob
NOTES_BLOB = '# Travel notes\n- Mumbai is humid in July.\n- INR/USD typically near 83.'
DemoServer.register_resource(ResourceSpec(
    uri='notes://travel.md',
    name='Travel notes',
    mimeType='text/markdown',
    reader=lambda: NOTES_BLOB,
))

# Prompt: a parameterised template
DemoServer.register_prompt(PromptSpec(
    name='brief_summary',
    description='Render a short briefing prompt for a topic.',
    arguments=[{'name': 'topic', 'description': 'Subject to brief on', 'required': True}],
    template='Give me a 3-bullet briefing on {topic}. Keep it under 60 words.',
))

print('=== DemoServer catalog published ===')
print(json.dumps({
    'tools':     DemoServer.list_tools(),
    'resources': DemoServer.list_resources(),
    'prompts':   DemoServer.list_prompts(),
}, indent=2))

=== DemoServer catalog published ===
{
  "tools": [
    {
      "name": "weather_lookup",
      "description": "Look up the current weather for a city.",
      "inputSchema": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "City name to get weather for, e.g. Mumbai"
          }
        },
        "required": [
          "city"
        ]
      }
    },
    {
      "name": "currency_convert",
      "description": "Convert an amount from one currency to another.",
      "inputSchema": {
        "type": "object",
        "properties": {
          "amount": {
            "type": "number"
          },
          "from_ccy": {
            "type": "string",
            "description": "ISO 4217, e.g. USD"
          },
          "to_ccy": {
            "type": "string",
            "description": "ISO 4217, e.g. INR"
          }
        },
        "required": [
          "amount",
          "from_ccy",
          "to_

## An `MCPClient` that introspects and calls the server

Now let's write the consumer side. The client knows nothing about the server's internals, only the protocol methods. That ignorance is the whole point.

Now, we will
- implement an `MCPClient` that holds a reference to a server and exposes the same five protocol methods,
- have the client call `list_tools()` and pretty-print the catalog as if it had just connected,
- call `call_tool('weather_lookup', {'city': 'Mumbai'})` and `read_resource('notes://travel.md')` end-to-end,
- finish with a `=== CLIENT TRACE ===` print so we can see every hop.

In [12]:
class MCPClient:
    """Thin client over an MCPServer. In a real deployment the server would be remote;
    here it is in-process. The METHOD SHAPE is what travels over the wire — keep it stable
    and the transport becomes a swap."""

    def __init__(self, server: MCPServer):
        self.server = server

    def list_tools(self) -> List[Dict[str, Any]]:
        return self.server.list_tools()

    def call_tool(self, name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        return self.server.call_tool(name, args)

    def list_resources(self) -> List[Dict[str, Any]]:
        return self.server.list_resources()

    def read_resource(self, uri: str) -> Dict[str, Any]:
        return self.server.read_resource(uri)

    def list_prompts(self) -> List[Dict[str, Any]]:
        return self.server.list_prompts()

client = MCPClient(DemoServer)
print(f'Connected to server: {client.server.name}')

Connected to server: demo-server


Now let's exercise the client. First, discover the catalog. A real client does this on connection so the LLM host can build its tool roster dynamically; no hardcoded names anywhere on the client side.

In [13]:
discovered = client.list_tools()
print('=== TOOLS DISCOVERED ===')
for t in discovered:
    print(f"  - {t['name']:<18} : {t['description']}")
    print(f"    arguments: {list(t['inputSchema']['properties'].keys())}")

=== TOOLS DISCOVERED ===
  - weather_lookup     : Look up the current weather for a city.
    arguments: ['city']
  - currency_convert   : Convert an amount from one currency to another.
    arguments: ['amount', 'from_ccy', 'to_ccy']


Then the actual calls. We invoke a tool, read a resource, and inspect both responses. The dicts are exactly what would come back across a JSON-RPC wire.

In [14]:
weather = client.call_tool('weather_lookup', {'city': 'Mumbai'})
fx      = client.call_tool('currency_convert', {'amount': 100, 'from_ccy': 'USD', 'to_ccy': 'INR'})
notes   = client.read_resource('notes://travel.md')

print('=== CLIENT TRACE ===')
print('weather :', json.dumps(weather, indent=2))
print('fx      :', json.dumps(fx, indent=2))
print('notes   :', json.dumps(notes, indent=2))

=== CLIENT TRACE ===
weather : {
  "tool": "weather_lookup",
  "isError": false,
  "content": "Mumbai: 32C, humid, light rain."
}
fx      : {
  "tool": "currency_convert",
  "isError": false,
  "content": "100 USD = 8300.00 INR"
}
notes   : {
  "uri": "notes://travel.md",
  "mimeType": "text/markdown",
  "text": "# Travel notes\n- Mumbai is humid in July.\n- INR/USD typically near 83."
}


## The JSON-RPC envelope on the wire

Now let's see what an MCP call actually looks like on the wire. The transport is plain **JSON-RPC 2.0**.

Now, we will
- show the request shape `{"jsonrpc": "2.0", "method": "tools/call", "params": {...}, "id": 1}`,
- show the matching response shape with `result` populated,
- explain why a standard envelope (not a bespoke schema) is what makes cross-host **composab**ility realistic,
- print both messages as raw JSON so we can read them.

In [15]:
def jsonrpc_request(method: str, params: Dict[str, Any], req_id: int = 1) -> Dict[str, Any]:
    """Wrap a server method call in the JSON-RPC 2.0 request envelope.
    This is exactly what a remote MCP client would put on the socket."""
    return {'jsonrpc': '2.0', 'method': method, 'params': params, 'id': req_id}

def jsonrpc_response(result: Any, req_id: int = 1) -> Dict[str, Any]:
    """Wrap a server reply in the JSON-RPC 2.0 response envelope."""
    return {'jsonrpc': '2.0', 'result': result, 'id': req_id}

wire_request  = jsonrpc_request('tools/call',
                                {'name': 'weather_lookup', 'arguments': {'city': 'Mumbai'}})
server_reply  = DemoServer.call_tool('weather_lookup', {'city': 'Mumbai'})
wire_response = jsonrpc_response(server_reply)

print('=== JSON-RPC REQUEST (client -> server) ===')
print(json.dumps(wire_request, indent=2))
print('=== JSON-RPC RESPONSE (server -> client) ===')
print(json.dumps(wire_response, indent=2))

=== JSON-RPC REQUEST (client -> server) ===
{
  "jsonrpc": "2.0",
  "method": "tools/call",
  "params": {
    "name": "weather_lookup",
    "arguments": {
      "city": "Mumbai"
    }
  },
  "id": 1
}
=== JSON-RPC RESPONSE (server -> client) ===
{
  "jsonrpc": "2.0",
  "result": {
    "tool": "weather_lookup",
    "isError": false,
    "content": "Mumbai: 32C, humid, light rain."
  },
  "id": 1
}


Once both sides agree on this envelope, transport choice (stdio subprocess, SSE, HTTP, websockets) is an integration detail, the method names and result shapes are stable. That stability is what enables the multi-team, multi-vendor tool catalogs MCP is designed for.

## Bridging to a LangChain agent

Now let's make the same MCP server usable from a LangChain agent. The bridge is small: walk the MCP tool catalog and emit a `@tool`-decorated wrapper for each entry. The wrapper just delegates to `server.call_tool(name, args)`.

Now, we will
- write a `mcp_to_langchain_tools(server)` helper that turns each MCP descriptor into a `@tool`,
- bind the resulting tools to the LLM with `llm.bind_tools([...])`,
- invoke on a multi-tool query and print the LLM's `tool_calls` so we can see which tools it picked,
- close by underlining that the SAME `DemoServer` could be plugged into any other client unchanged.

In [ ]:
from pydantic import create_model
from typing import Optional

def mcp_to_langchain_tools(server: MCPServer):
    _type_map = {'string': str, 'number': float, 'integer': int, 'boolean': bool}

    bridges = []
    for spec in server.list_tools():
        name       = spec['name']
        desc       = spec['description']
        properties = spec['inputSchema'].get('properties', {})
        required   = set(spec['inputSchema'].get('required', []))

        field_defs = {}
        for prop_name, prop_info in properties.items():
            py_type = _type_map.get(prop_info.get('type', 'string'), str)
            field_defs[prop_name] = (py_type, ...) if prop_name in required else (Optional[py_type], None)

        ArgsSchema = create_model(f'{name}_args', **field_defs)


# make_bridge('weather_lookup', 'Look up...', ArgsSchema) — this is a closure factory. 
# It defines _bridge(**kwargs) and immediately decorates it with @tool(name_or_callable='weather_lookup', description=..., args_schema=ArgsSchema), 
# turning it into a proper LangChain Tool object.

# Why the closure/factory pattern (make_bridge) is necessary, not just a stylistic choice: 
# if you defined _bridge directly inside the for spec in server.list_tools() loop without wrapping it in make_bridge, every _bridge closure
#  would capture the loop variable name by reference, not by value — so by the time any tool is actually called, name would have already changed to
#  whichever tool was processed last in the loop (a classic Python closure-in-a-loop bug).
# Wrapping the closure creation inside make_bridge(_name, _desc, _schema) forces each call to get its own local copies of _name/_desc/_schema, 
# correctly pinned to that iteration's values. 
# This is a subtle but important correctness detail — good to notice since it's easy to get wrong.
        def make_bridge(_name, _desc, _schema):
            @tool(name_or_callable=_name, description=_desc, args_schema=_schema)
            def _bridge(**kwargs) -> str:
                response = server.call_tool(_name, kwargs)
                return str(response['content'])
            return _bridge

        bridges.append(make_bridge(name, desc, ArgsSchema))
    return bridges

lc_tools = mcp_to_langchain_tools(DemoServer)
print('Bridged MCP tools to LangChain:', [t.name for t in lc_tools])

Bridged MCP tools to LangChain: ['weather_lookup', 'currency_convert']


And now the actual bind. `llm.bind_tools(lc_tools)` returns a new chat model whose responses can carry `tool_calls`. We invoke on a query that needs both tools so we can see the LLM pick more than one.

In [14]:
llm_with_tools = llm.bind_tools(lc_tools)

user_query = ('How much is 100 USD in INR.')
ai_message = llm_with_tools.invoke(user_query)

print('=== LLM tool_calls ===')
for call in ai_message.tool_calls:
    print(f"  - {call['name']}({call['args']})")

print('---- executing each tool_call against the SAME MCPServer ----')
for call in ai_message.tool_calls:
    out = DemoServer.call_tool(call['name'], call['args'])
    print(f"  {call['name']:<18} -> {out['content']}")

=== LLM tool_calls ===
  - currency_convert({'from_ccy': 'USD', 'amount': 100, 'to_ccy': 'INR'})
---- executing each tool_call against the SAME MCPServer ----
  currency_convert   -> 100 USD = 8300.00 INR


*Tip: the `DemoServer` we built is unchanged from the previous section; the same catalog, the same handlers. The LangChain agent is just one possible client. Swap the agent for a different framework, or a CLI, or another LLM host, and the server still works. That **reuse** is the operational win MCP delivers.*

## Tips and common pitfalls

**BENEFITS**
  - Composability: any MCP client can use any MCP server without per-tool glue.
  - Reuse: a tool written once is consumable by every agent, framework, and LLM host.
  - Multi-tenant catalog: teams publish servers, an org-wide registry indexes them.
  - Transport-agnostic: stdio, SSE, HTTP all carry the same JSON-RPC envelope.
  - Discovery at runtime: list_tools / list_resources / list_prompts means no hardcoded names.

**LIMITATIONS**
  - Transport latency: each call adds JSON-RPC + IPC / network overhead vs in-process.
  - Auth and security: stdio has none built in; SSE / HTTP need explicit token plumbing.
  - Schema drift: a server can change its inputSchema between deploys; clients must handle missing fields gracefully.
  - Operational surface: each server is a process to monitor, version, and roll back.

**WHEN TO USE MCP**

- Multi-agent / multi-team       -> Yes - MCP. Cross-team reuse is the point.
- Multi-host (IDE + agent)       -> Yes - MCP. Shared tool surface across processes.
- Single-team single-agent       -> No  - native @tool is simpler and faster.
- Sub-second latency budget      -> Maybe - measure the JSON-RPC overhead first.

### Adopting MCP in production
- Pin server versions per-deploy and surface the version through `list_tools()` metadata. The protocol does not version your tools for you, schema drift between server versions is the most common production breakage.
- Use stdio transport only inside a trust boundary. For anything cross-host, use SSE or HTTP with a real token check inside each tool body, stdio has no built-in auth.
- Treat resources as read-only and idempotent. The spec does not forbid side effects but every client assumes `read_resource` is safe to call many times.
- Keep `temperature=0` on the LLM when binding tools. Higher temperature increases the rate of malformed tool argument JSON, which forces a retry round-trip.

### Common pitfalls when wrapping MCP tools
- A `@tool` bridge that swallows server errors will hide schema mismatches forever. Let the exception propagate to the agent loop so the LLM can react. <br>Mitigation: return `{'isError': True, 'content': ...}` from `call_tool` rather than raising silently.
- Do not hardcode tool names on the client. Always discover via `list_tools()` so a server upgrade is a no-code change on the client. The whole point of the protocol is that names are runtime data, not source code.
- Watch the round-trip count. A multi-tool query can fan out to N JSON-RPC calls per turn; for high-throughput agents, batch where the spec allows it or move hot tools in-process.